In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [22]:
# importing the cleaned dataset, converting date column to datetime format and sorting chronologically.

df = pd.read_csv("data_cleaned.csv")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

df.head()

,date,painkiller_usage,study_hours_daily,exam_period,coffee_consumption,is_period
0,2025-06-01,0,5.0,0,1,0
1,2025-06-02,1,7.0,1,1,0
2,2025-06-03,0,4.0,1,1,0
3,2025-06-04,0,8.0,1,1,0
4,2025-06-05,0,4.0,1,1,0


In [23]:
# seperating the target variable(study_hours_daily) and the feature variables used for prediction.

X = df.drop(columns=["study_hours_daily", "date"])
y = df["study_hours_daily"]

X.head()


,painkiller_usage,exam_period,coffee_consumption,is_period
0,0,0,1,0
1,1,1,1,0
2,0,1,1,0
3,0,1,1,0
4,0,1,1,0


In [24]:
# Splitting the data into training and testing sets
# Since the data is time-ordered, the split is performed without shuffling

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

X_train.shape, X_test.shape


((143, 4), (36, 4))

In [25]:
# Training a Multiple Linear Regression model
# Making predictions on the test set and evaluating model performance using MAE, RMSE, and R² score


lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2 = r2_score(y_test, y_pred_lr)

print(f"Mean Absolute Error: {lr_mae}")
print(f"Root Mean Squared Error: {lr_rmse}")
print(f"R^2 Score: {lr_r2}")



Mean Absolute Error: 2.3156828851466527
Root Mean Squared Error: 2.765296827260713
R^2 Score: -0.5431805113501995


In [26]:
# Checkinhg the regression coefficients for each feature to understand the magnitude of it's effect
pd.Series(lr.coef_, index=X.columns)


painkiller_usage      0.460872
exam_period           4.930136
coffee_consumption    0.321030
is_period            -0.971758
dtype: float64

In [27]:
# Training a Decision Tree Regressor with a limited depth to capture non-linear relationships while avoiding overfitting
# Evaluating the Decision Tree model using the same performance metrics for fair comparison

dt = DecisionTreeRegressor(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

dt_mae = mean_absolute_error(y_test, y_pred_dt)
dt_rmse = np.sqrt(mean_squared_error(y_test, y_pred_dt))
dt_r2 = r2_score(y_test, y_pred_dt)

print(f"Mean Absolute Error: {dt_mae}")
print(f"Root Mean Squared Error: {dt_rmse}")
print(f"R^2 Score: {dt_r2}")


Mean Absolute Error: 2.0320126977109734
Root Mean Squared Error: 2.668448588286518
R^2 Score: -0.43698058042335375


In [28]:
# checking the feature importance scores to identify the most influential predictors

pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=False)


exam_period           0.849576
coffee_consumption    0.059558
is_period             0.053773
painkiller_usage      0.037093
dtype: float64

In [29]:
# Training the Random Forest model and evaluating its predictive performance on unseen test data

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)

print(f"Mean Absolute Error: {rf_mae}")
print(f"Root Mean Squared Error: {rf_rmse}")
print(f"R^2 Score: {rf_r2}")



Mean Absolute Error: 2.028289811505882
Root Mean Squared Error: 2.6653528771635013
R^2 Score: -0.4336483847720396


In [30]:
# Analyzing feature importance from the Random Forest model, which provides the most reliable importance estimates

pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)


exam_period           0.806135
coffee_consumption    0.082375
is_period             0.063906
painkiller_usage      0.047584
dtype: float64